
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>



<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">

<div style="background: #1B5162; color: white; border-radius: 8px; padding: 28px 32px; text-align: center; position: relative;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Lesson 14</div>
  <div style="font-size: 24pt; font-weight: 700; line-height: 1.3;">Automate Your Pipeline with a Lakeflow Job</div>
  <div style="font-size: 14pt; margin-top: 12px; opacity: 0.9;">Create a multi-task Lakeflow Job that orchestrates the Bronze → Silver → Gold pipeline with task dependencies, and monitor its execution.</div>
</div>

</div>

## REQUIRED — SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before running this notebook, confirm your compute environment at the top-right of the notebook.

- Click the compute dropdown and select **Serverless** (the default option).
- If you do not see Serverless available, contact your workspace administrator.

**Note:** This notebook was developed and tested on **Serverless compute**. Other compute options may work but are not guaranteed to behave the same.
  </div>
</div>

### Setup
Run the cell below to configure your environment.

In [0]:
%run ./Includes/Classroom-Setup-1

**In this lesson:** In Lesson 12, you learned how to build the full Medallion Architecture pipeline manually: Bronze, Silver, Gold. In this lesson, you'll automate it so it runs on its own using Lakeflow Jobs.


<!-- LEARN: Lakeflow Jobs -->
<!-- Template: source-process-output-flow (adapted) -->

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">

<div style="font-size: 20pt; font-weight: 700; color: #0b2026; margin-bottom: 6px;">Lakeflow Jobs: Orchestrate Your Pipeline</div>
<div style="font-size: 14pt; color: #5A6F77; margin-bottom: 24px;">A Lakeflow Job runs one or more notebooks as tasks, with dependencies between them. You define what runs, in what order, and on what schedule.</div>

<div style="display: flex; justify-content: center; align-items: center; gap: 16px;">

  <!-- Task 1 -->
  <div style="flex: 0 0 260px; background: #F9F7F4; border-radius: 10px; padding: 18px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center; border-top: 6px solid #CD7F32;">
    <div style="font-size: 16pt; font-weight: 700; margin-bottom: 8px;">Task 1: Setup + Bronze</div>
    <div style="font-size: 14pt; color: #5A6F77;">Creates the volume, loads CSV files into the Bronze table</div>
  </div>

  <!-- Arrow -->
  <div style="font-size: 28pt; color: #618794;">→</div>

  <!-- Task 2 -->
  <div style="flex: 0 0 260px; background: #F9F7F4; border-radius: 10px; padding: 18px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center; border-top: 6px solid #FFAB00;">
    <div style="font-size: 16pt; font-weight: 700; margin-bottom: 8px;">Task 2: Silver + Gold</div>
    <div style="font-size: 14pt; color: #5A6F77;">Transforms Bronze → Silver, aggregates Silver → Gold</div>
  </div>

</div>

<!-- Connector line -->
<div style="margin: 14px auto 0 auto; width: 60%; height: 4px; background: #1B5162; border-radius: 2px;"></div>

<!-- Key point -->
<div style="margin-top: 18px; padding: 16px 20px; background: #FFF6F4; border: 3px solid #FF5F46; border-radius: 10px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <strong>The dependency arrow means Task 2 only runs if Task 1 succeeds.</strong> If Bronze fails, Silver and Gold won't run with stale or missing data. This is how production pipelines stay reliable.
  </div>
</div>

</div>

##### EXPAND FOR ADDITIONAL NOTES

<details>

**What Lakeflow Jobs give you**

- **Task dependencies:** Define which tasks must complete before others start. The job visualizes this as a directed acyclic graph (DAG).
- **Scheduling:** Run on a cron schedule (every hour, daily at 6am), on file arrival (new files land in a volume), or on table updates.
- **Monitoring:** Each run shows status per task (Pending → Running → Succeeded/Failed), duration, and output.
- **Alerting:** Configure email or Slack notifications on failure so your team knows immediately when a pipeline breaks.

**Jobs vs. notebooks**

- Running a notebook interactively is great for development and exploration. But production pipelines need to run automatically, on a schedule, without someone clicking "Run All."
- A Lakeflow Job wraps your notebooks into an automated, monitored, and schedulable workflow. The notebooks are the same ones you developed interactively.

</details>

### Explore: Locate the task notebooks

Two task notebooks have been pre-loaded in the `Resources` folder of this project. These are the notebooks the job will execute:

- **Task 1 - Setup - Bronze** — Sets up the environment and loads CSV files into a Bronze table
- **Task 2 - Silver - Gold** — Transforms Bronze into Silver and aggregates into Gold

You can open them from the file browser to review their contents before creating the job.

### Explore: Create the Lakeflow Job

Now you'll create a job that runs these two notebooks in sequence.

##### **Follow these steps:**

1. In the left sidebar, right-click **Jobs & Pipelines** and select **Open in new tab**
2. Click **Create** and select **Job**
3. Name your job (e.g., `yourname-bronze-silver-gold-pipeline`)

##### **Configure Task 1:**
4. Set the task name to **Setup-Bronze**
5. Type: **Notebook**
6. Source: **Workspace**
7. Path: Navigate to your project folder → `Resources` → select **Task 1 - Setup - Bronze**
8. Compute: **Serverless**
9. Click **Create task**

##### **Configure Task 2:**
10. Click **Add task** → **Notebook**
11. Set the task name to **Silver-Gold**
12. Path: Navigate to `Resources` → select **Task 2 - Silver - Gold**
13. Compute: **Serverless**
14. Depends on: **Setup-Bronze**
15. Run if dependencies: **All succeeded**
16. Click **Create task**

You should see a visual DAG (directed acyclic graph) with a dependency arrow from Setup-Bronze to Silver-Gold.

### Explore: Browse scheduling options

Before running the job, take a moment to explore what scheduling options are available:

1. In the job editor, find **Schedules & Triggers** on the right side
2. Click **Add trigger**
3. Browse the options: **Scheduled** (cron), **File arrival**, **Table updates**, **Continuous**
4. Click **Cancel**. We'll run the job manually for now

In production, you'd schedule this to run daily or trigger it when new files arrive in the volume.

### Explore: Run the job

1. Click **Run now** in the top right
2. Click the **Runs** tab to watch the execution
3. You'll see each task progress: **Pending** → **Running** → **Succeeded**
4. The job typically takes 2-5 minutes to complete

Once both tasks show **Succeeded**, click either task to see its output.

### Explore: Verify the results

After the job completes, verify that the pipeline created the expected tables. The job creates tables with a `_job` suffix to avoid overwriting your manual tables from Lesson 12.

In [0]:
%sql
SELECT * 
FROM current_employees_bronze_job;

In [0]:
%sql
SELECT * 
FROM current_employees_silver_job;

In [0]:
%sql
SELECT * 
FROM total_roles_gold_job;

The same Bronze → Silver → Gold pipeline you built manually in Lesson 12, now running automatically as an orchestrated job.


<!-- Micro-win summary -->

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">
<div style="margin-top: 10px; padding: 18px 24px; background: #FFF6F4; border: 3px solid #FF5F46; border-radius: 10px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <div style="font-weight: 700; margin-bottom: 8px;">What you just did:</div>
    <ul style="padding-left: 20px; margin: 0;">
      <li>Created a <strong>Lakeflow Job</strong> with two tasks and a dependency</li>
      <li>Explored scheduling options (cron, file arrival, table updates)</li>
      <li>Ran the job and monitored each task's execution</li>
      <li>Verified the automated pipeline produced the same Bronze → Silver → Gold results</li>
    </ul>
  </div>
</div>
</div>


&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>